# Intelligent Traffic & Video Analytics - Google Colab (GPU)

This notebook is only a **thin wrapper** around the project CLI. The Python package does
not need to be converted to notebook code.

Before running:

1. Select **Runtime > Change runtime type > T4 GPU** (or another GPU).
2. Put the project zip in Google Drive, or clone your repository into `/content`.
3. Put a traffic video in Google Drive (recommended) or upload it.
4. Run the cells from top to bottom.

Colab storage is temporary, so the last cells copy the results to Google Drive.


In [ ]:
!nvidia-smi
import torch

print("\ntorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: enable Runtime > Change runtime type > GPU")

## 1. Get the project code

Option A (recommended): upload `traffic-video-analytics.zip` to
`/content/drive/MyDrive/` and keep `ZIP_PATH` below.

Option B: clone your repository instead:

```python
!git clone https://github.com/<your-user>/traffic-video-analytics.git /content/traffic-video-analytics
```

The package itself is unchanged - only the working directory matters.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import zipfile

PROJECT_DIR = "/content/traffic-video-analytics"
ZIP_PATH = "/content/drive/MyDrive/traffic-video-analytics.zip"  # <-- edit if needed

if not os.path.isdir(PROJECT_DIR):
    if os.path.exists(ZIP_PATH):
        with zipfile.ZipFile(ZIP_PATH) as archive:
            archive.extractall("/content")
    else:
        raise FileNotFoundError(
            f"Upload the project zip to {ZIP_PATH}, or clone your repo into {PROJECT_DIR}."
        )

%cd /content/traffic-video-analytics
!ls

## 2. Install dependencies

Colab already ships CUDA-enabled PyTorch, so do **not** use the CPU PyTorch command from
the README here. Install Ultralytics and the project itself without changing the existing
torch build.


In [ ]:
!pip install -q ultralytics
!pip install -q -e . --no-deps
!traffic-analytics doctor

## 3. Select the input video

Recommended: upload the clip to Google Drive and point `VIDEO_PATH` to it. For a quick
test, upload directly with `google.colab.files.upload()`.


In [ ]:
import os

VIDEO_PATH = "/content/drive/MyDrive/traffic-sample.mp4"  # <-- edit

# Alternative: upload a video from your computer.
# from google.colab import files
# uploaded = files.upload()
# VIDEO_PATH = "/content/" + next(iter(uploaded))

assert os.path.exists(VIDEO_PATH), f"Video not found: {VIDEO_PATH}"
print("Using:", VIDEO_PATH)

## 4. Configure GPU, stride, and counting line

Normalized line coordinates (`0.0 - 1.0`) are resolution independent. A horizontal
left-to-right line has its positive side below it (southbound by default).

Inspect the next cell's preview image and adjust the line to match your camera.


In [ ]:
import yaml

CONFIG_PATH = "config/config.yaml"
with open(CONFIG_PATH, encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

config["video"]["path"] = VIDEO_PATH
config["video"]["frame_stride"] = 1          # GPU can process every frame
config["model"]["device"] = "0"               # first CUDA GPU
config["model"]["imgsz"] = 640

line = config["lines"][0]
line["start"] = [0.15, 0.55]                  # <-- calibrate
line["end"] = [0.85, 0.55]                    # <-- calibrate
line["positive_direction"] = "southbound"
line["negative_direction"] = "northbound"

with open(CONFIG_PATH, "w", encoding="utf-8") as handle:
    yaml.safe_dump(config, handle, sort_keys=False)

print("Updated", CONFIG_PATH)

In [ ]:
!traffic-analytics preview-lines --config config/config.yaml --frame 0 --out output/colab_line_preview.jpg

from IPython.display import Image, display
display(Image(filename="output/colab_line_preview.jpg"))

If the line or arrows are wrong, edit the coordinates in the previous cell, rerun it,
then rerun the preview. Swap `positive_direction` / `negative_direction` if traffic
directions are inverted.

## 5. Run YOLO + ByteTrack on the GPU


In [ ]:
!traffic-analytics analyze --config config/config.yaml --device 0

## 6. Play the annotated result

Browsers do not reliably play the OpenCV `mp4v` codec, so the annotated file is re-encoded
to H.264 for inline playback. For long clips, set `embed=False` or download the file
instead of embedding it.


In [ ]:
from pathlib import Path
import subprocess

videos = sorted(
    Path("output/videos").glob("*_annotated.mp4"),
    key=lambda path: path.stat().st_mtime,
)
assert videos, "No annotated video found - run the analysis cell first."
video = videos[-1]
browser_video = video.with_name(video.stem + "_h264.mp4")

subprocess.run(
    [
        "ffmpeg", "-y", "-loglevel", "error", "-i", str(video),
        "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart",
        str(browser_video),
    ],
    check=True,
)

from IPython.display import Video
print(browser_video)
Video(str(browser_video), embed=True, width=960)

## 7. Show the traffic report

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

reports = sorted(
    Path("output/reports").glob("*/summary.md"),
    key=lambda path: path.stat().st_mtime,
)
assert reports, "No report found - run the analysis cell first."
display(Markdown(reports[-1].read_text(encoding="utf-8")))

## 8. Save results to Google Drive

Colab deletes `/content` when the runtime ends. Copy `output/` to Drive before closing.


In [ ]:
import shutil
from datetime import datetime
from pathlib import Path

destination = (
    Path("/content/drive/MyDrive/traffic-analytics-output")
    / datetime.now().strftime("%Y%m%d_%H%M%S")
)
shutil.copytree("output", destination, dirs_exist_ok=True)
print("Saved to", destination)

Optional: download a zip of all outputs instead.

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/traffic_analytics_output", "zip", "output")
files.download(archive)

## Notes

- The offline synthetic demo also works on Colab (CPU is enough):

  ```python
  !traffic-analytics demo --output-dir output/demo
  ```

- For long videos, use `--max-frames` or process the file in chunks to stay inside the
  Colab session limit. Outputs go to `output/` unless overridden.
- A GPU accelerates YOLO, but video decoding and OpenCV annotation still use CPU. If the
  bottleneck is annotation, pass `--no-video` for pure data generation.
- Use footage you have the right to process. Ultralytics YOLO has its own license terms.
